# [실습 06] 프레임워크 맛보기 — 순차 파이프라인 직접 만들기

> **연계**: 제3부 06장(프레임워크) · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-1.5B-Instruct`

**학습 목표**
- LangGraph·CrewAI 같은 프레임워크의 **순차 파이프라인(노드 연결)** 개념을 코드로 이해한다.
- 각 단계가 앞 단계의 출력을 입력으로 받는 구조를 구현한다.

> 참고: 실제 LangChain 실습은 저장소의 `07_2_Chain_QA_Agent_Practice.ipynb`를 함께 보세요.

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import pipeline
gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")
def step(system, user):
    msg = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return gen(msg, max_new_tokens=200, do_sample=False)[0]["generated_text"][-1]["content"].strip()

## 1. 노드(단계) 정의 → 파이프라인으로 연결

`추출 → 요약 → 번역` 세 노드를 순차로 연결합니다. (05-2 순차 토폴로지)

In [ ]:
text = "AI 에이전트는 LLM을 두뇌로 삼아 도구와 기억을 결합해 스스로 목표를 수행하는 시스템이다. 최근 소프트웨어 개발과 연구에서 활용이 늘고 있다."

def node_keyphrase(x): return step("핵심 키워드 3개만 콤마로 뽑아라.", x)
def node_summary(x):   return step("한 문장으로 요약하라.", x)
def node_translate(x): return step("영어로 번역하라.", x)

PIPELINE = [node_keyphrase, node_summary, node_translate]

cur = text
for i, node in enumerate(PIPELINE, 1):
    cur = node(cur)
    print(f"[단계 {i}] {node.__name__} → {cur}\n")

## 2. 정리
- 프레임워크의 핵심인 **노드 연결(파이프라인)** 을 직접 구현했다.
- 각 단계는 앞 단계 출력을 입력으로 받는다(06-1 상태 전달).
- **더 해보기**: 노드 순서를 바꾸거나 `node_sentiment`를 추가해 보세요.